# Inventory Aging Analysis

## Project Overview
Analyze stock records to understand aged inventory, slow movers, write-off risks, and category-level stocking issues. This is a descriptive analytics project.

## Learning Objectives
- Compute inventory aging buckets and identify slow-moving stock
- Calculate days of supply and turnover ratios
- Identify write-off risk by category and product
- Visualize aging distributions and trends

## Problem Statement
Finance and operations need to understand how much capital is tied up in aged inventory, which categories have slow movers, and where write-off risk is highest.

## Why This Project Matters
Aged inventory ties up working capital, occupies warehouse space, and risks obsolescence. Identifying slow movers enables clearance strategies and better demand planning.

## Dataset Overview
Synthetic inventory snapshot: ~1,000 SKUs across 6 categories with receipt dates, quantities, unit costs, and recent demand data.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 1000
categories = np.random.choice(['Electronics', 'Apparel', 'Food & Bev', 'Home & Garden', 'Auto Parts', 'Industrial'],
                                n, p=[0.20, 0.18, 0.15, 0.17, 0.15, 0.15])
snapshot_date = pd.Timestamp('2024-12-31')

# Receipt dates — spread over past 2 years with some very old stock
receipt_dates = []
for _ in range(n):
    if np.random.random() < 0.08:
        days_ago = np.random.randint(365, 730)  # old stock
    elif np.random.random() < 0.3:
        days_ago = np.random.randint(120, 365)
    else:
        days_ago = np.random.randint(5, 120)
    receipt_dates.append(snapshot_date - pd.Timedelta(days=int(days_ago)))

qty_on_hand = np.clip(np.random.lognormal(4, 1.5, n).astype(int), 1, 5000)
unit_cost = np.clip(np.random.lognormal(2.5, 1.0, n), 1, 500).round(2)
avg_daily_demand = np.clip(np.random.lognormal(1.0, 1.2, n), 0.01, 200).round(2)

df = pd.DataFrame({
    'SKU': [f'SKU-{i:05d}' for i in range(n)],
    'Category': categories,
    'ReceiptDate': receipt_dates,
    'QtyOnHand': qty_on_hand,
    'UnitCost': unit_cost,
    'AvgDailyDemand': avg_daily_demand,
})
df['StockValue'] = (df['QtyOnHand'] * df['UnitCost']).round(2)
df['AgeDays'] = (snapshot_date - df['ReceiptDate']).dt.days
df['DaysOfSupply'] = (df['QtyOnHand'] / df['AvgDailyDemand']).round(1)
df['AgingBucket'] = pd.cut(df['AgeDays'], bins=[0, 30, 90, 180, 365, 9999],
                            labels=['0-30', '31-90', '91-180', '181-365', '365+'])
df['Turnover'] = (df['AvgDailyDemand'] * 365 / df['QtyOnHand']).round(2)
# Slow mover: days of supply > 180
df['SlowMover'] = (df['DaysOfSupply'] > 180).astype(int)

print(f'Dataset shape: {df.shape}')
print(f'Total stock value: ${df["StockValue"].sum():,.0f}')
print(f'Slow movers: {df["SlowMover"].mean():.1%}')
print(f'\nAging bucket distribution:\n{df["AgingBucket"].value_counts().sort_index()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'SKU count: {df["SKU"].nunique()}')
print(f'Age range: {df["AgeDays"].min()} - {df["AgeDays"].max()} days')
print(f'Stock value range: ${df["StockValue"].min():,.2f} - ${df["StockValue"].max():,.2f}')
print(f'\nCategory distribution:\n{df["Category"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

aging_value = df.groupby('AgingBucket')['StockValue'].sum()
aging_value.plot.bar(ax=axes[0,0], edgecolor='black', color='coral')
axes[0,0].set_title('Stock Value by Aging Bucket')
axes[0,0].set_ylabel('Total Value ($)')
axes[0,0].tick_params(axis='x', rotation=0)

df.groupby('Category')['StockValue'].sum().sort_values().plot.barh(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Stock Value by Category')

df['AgeDays'].hist(bins=50, ax=axes[1,0], edgecolor='black', alpha=0.7, color='green')
axes[1,0].set_title('Age Distribution (Days)')
axes[1,0].axvline(180, color='red', linestyle='--', label='180 days')
axes[1,0].axvline(365, color='darkred', linestyle='--', label='365 days')
axes[1,0].legend()

df.groupby('Category')['SlowMover'].mean().sort_values().plot.barh(ax=axes[1,1], edgecolor='black', color='orange')
axes[1,1].set_title('Slow Mover % by Category')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Aging Analysis by Category

In [ ]:
aging_cat = df.groupby(['Category', 'AgingBucket'])['StockValue'].sum().unstack(fill_value=0)
aging_cat_pct = aging_cat.div(aging_cat.sum(axis=1), axis=0).round(3)
print('Stock Value % by Aging — Category Breakdown:')
print(aging_cat_pct)

fig, ax = plt.subplots(figsize=(12, 6))
aging_cat_pct.plot.bar(stacked=True, ax=ax, edgecolor='black')
ax.set_title('Aging Distribution by Category (% of Value)')
ax.set_ylabel('Proportion')
ax.legend(title='Aging Bucket', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'aging_category.png'), dpi=100, bbox_inches='tight')
plt.show()

## Write-Off Risk Assessment

In [ ]:
# Define write-off risk: 365+ days old items
risk = df[df['AgeDays'] >= 365].copy()
risk_summary = risk.groupby('Category').agg(
    skus=('SKU', 'count'),
    total_value=('StockValue', 'sum'),
    avg_age=('AgeDays', 'mean'),
    avg_demand=('AvgDailyDemand', 'mean'),
).round(2).sort_values('total_value', ascending=False)
print(f'Write-off risk (365+ days): {len(risk)} SKUs, ${risk["StockValue"].sum():,.0f} value')
print('\nBy Category:')
print(risk_summary)

fig, ax = plt.subplots(figsize=(10, 5))
risk_summary['total_value'].plot.bar(ax=ax, edgecolor='black', color='red', alpha=0.6)
ax.set_title('Write-Off Risk Value by Category (365+ Days)')
ax.set_ylabel('Value at Risk ($)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'writeoff_risk.png'), dpi=100, bbox_inches='tight')
plt.show()

## Days of Supply Analysis

In [ ]:
dos_cat = df.groupby('Category')['DaysOfSupply'].agg(['mean', 'median', 'max']).round(1)
dos_cat.columns = ['Mean DOS', 'Median DOS', 'Max DOS']
print('Days of Supply by Category:')
print(dos_cat)

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='Category', y='DaysOfSupply', ax=ax, showfliers=False)
ax.set_title('Days of Supply Distribution by Category')
ax.tick_params(axis='x', rotation=45)
ax.axhline(90, color='orange', linestyle='--', label='90-day target')
ax.axhline(180, color='red', linestyle='--', label='Slow mover threshold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'days_of_supply.png'), dpi=100, bbox_inches='tight')
plt.show()

## Turnover Analysis

In [ ]:
turn_cat = df.groupby('Category')['Turnover'].agg(['mean', 'median']).round(2)
print('Inventory Turnover by Category:')
print(turn_cat)

fig, ax = plt.subplots(figsize=(10, 5))
turn_cat['mean'].sort_values().plot.barh(ax=ax, edgecolor='black', color='teal')
ax.set_title('Avg Inventory Turnover by Category')
ax.set_xlabel('Turns per Year')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'turnover.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **365+ day old stock** represents significant write-off risk — these SKUs need clearance or disposal action
- **Food & Bev** and **Apparel** are particularly vulnerable to aging due to perishability and seasonality
- **Days of supply** exceeding 180 days flags overstocking — triggers should be set for reorder policy adjustments
- **Turnover rates** vary significantly by category — slow turners tie up capital disproportionately
- The Pareto principle applies: a small percentage of SKUs likely hold a large percentage of the aged value

## Limitations
- Single-point snapshot — no time series of inventory levels
- Uniform demand assumption (avg daily demand) — real demand is lumpy
- No lot tracking or expiration dates
- No procurement pipeline data (incoming POs)
- No seasonal demand adjustment

## How to Improve This Project
- Use real inventory data with multiple snapshots for trend analysis
- Add lot-level tracking with expiration dates for perishables
- Include demand forecasting to compute forward-looking days of supply
- Build ABC/XYZ classification for inventory management strategy
- Add cost-of-carry analysis to quantify the financial impact of aging

## Production Considerations
- Weekly aging reports with escalation for slow movers
- Automated clearance triggers based on aging thresholds
- Integration with demand planning for reorder optimization
- Dashboard with drill-down from category → SKU level

## Common Mistakes
- Using quantity instead of value for aging analysis (masks high-value risks)
- Not separating safety stock from excess when assessing overstocking
- Ignoring demand patterns when setting aging thresholds
- Treating all categories with the same aging policy

## Mini Challenge / Exercises
1. What % of total stock value is held in items older than 180 days?
2. Which category has the lowest turnover? What practical actions could improve it?
3. Identify the top 10 SKUs by stock value that are also 365+ days old. What action would you recommend?
4. If the company wrote off all 365+ day inventory, what % of total stock value would be lost?

## Final Summary / Key Takeaways
- Inventory aging analysis is essential for working capital optimization
- Aging buckets, days of supply, and turnover provide complementary views of inventory health
- Write-off risk identification enables proactive clearance strategies
- Category-level analysis reveals which areas need tighter inventory policies
- Regular aging reviews prevent capital from silently eroding in warehouses